# DEV · Compra Fácil Bajío: reseñas negativas
**Responsable:** Luis

**Especificaciones del cliente (prueba, umbral 0.5):** exhaustividad ≥ 0.90 · precisión ≥ 0.85. Modelo entrenado por nosotros.

In [ ]:
import random, numpy as np, tensorflow as tf
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

## 1. Datos (carga y prueba exactas del enunciado)

In [ ]:
import pandas as pd

URL = "https://huggingface.co/datasets/SetFit/amazon_reviews_multi_es/resolve/main/{}.jsonl"
partes = {p: pd.read_json(URL.format(p), lines=True) for p in ["train", "validation", "test"]}
for df in partes.values():
    df["estrellas"] = df["label"] + 1                  # label va de 0 a 4
train, val, test = partes["train"], partes["validation"], partes["test"]

# Prueba del cliente: la partición test sin las de 3 estrellas (4,000 reseñas, mitad negativas).
# 1 = negativa. No la toquen hasta el final.
prueba = test[test["estrellas"] != 3]
X_prueba, y_prueba = prueba["text"].values, (prueba["estrellas"] <= 2).astype(int).values

## 2. Exploración
Tamaños, distribución de estrellas, longitud por clase, ejemplos.

In [ ]:
import matplotlib.pyplot as plt
print({k: len(v) for k, v in partes.items()})
print(train["estrellas"].value_counts().sort_index())
# TODO: barras de estrellas, histograma de longitud por clase, ejemplos de 1, 3 y 5 estrellas

## 3. Preparación
**Decisión sobre las de 3 estrellas:** se quitan, porque son ambiguas (ni buenas ni malas) y la prueba no las incluye; dejarlas metería ruido en la frontera de decisión.

In [ ]:
tr = train[train["estrellas"] != 3]
va = val[val["estrellas"] != 3]
X_tr, y_tr = tr["text"].values, (tr["estrellas"] <= 2).astype(int).values
X_val, y_val = va["text"].values, (va["estrellas"] <= 2).astype(int).values
print(len(X_tr), len(X_val), y_tr.mean().round(3), y_val.mean().round(3))
# El vectorizador/tokenizador se ajusta SOLO con X_tr.

## 4. Métricas del cliente

In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score

def metricas(y_true, prob, umbral=0.5):
    pred = (prob >= umbral).astype(int)
    rec = recall_score(y_true, pred, zero_division=0)
    prec = precision_score(y_true, pred, zero_division=0)
    return {"exhaustividad": rec, "precision": prec, "exactitud": accuracy_score(y_true, pred),
            "cumple_metricas": rec >= 0.90 and prec >= 0.85}

## 5. Línea base

In [ ]:
from sklearn.dummy import DummyClassifier
base = DummyClassifier(strategy="most_frequent").fit(X_tr.reshape(-1, 1), y_tr)
print(metricas(y_val, base.predict_proba(X_val.reshape(-1, 1))[:, 1]))

## 6. Modelos
TF-IDF + red densa · Embedding + Bi-LSTM · BETO ajustado (Colab GPU). Elegir con validación.

In [ ]:
# TODO: definir, entrenar y comparar en validación

## 7. Evaluación única en prueba y `resultados`

In [ ]:
# TODO: evaluar UNA vez cada modelo congelado en X_prueba
# resultados = pd.DataFrame(filas, columns=["modelo","tipo","parametros","exhaustividad","precision","exactitud","cumple","elegido"])
# resultados.to_csv("resultados_resenas.csv", index=False)

## 8. Justificación de la elección

## 9. Análisis de 3 errores

In [ ]:
# TODO: mostrar el texto de 3 reseñas mal clasificadas y explicar por qué

## 10. Guardar el modelo elegido
Si pesa > 100 MB (BETO), subir a Hugging Face y poner el enlace en el README.

In [ ]:
# TODO